# 09. RAG Evaluation
## 검색과 답변을 최소 지표로 평가하기

### 이번 실습의 위치

```text
01~08 Advanced RAG 개선
        ↓
09 Evaluation
        ↓
정말 좋아졌는지 확인
```

이번 실습에서는 평가 지표를 많이 다루지 않는다.

딱 두 가지만 확인한다.

```text
Retrieval
→ Hit@K
→ 관련 근거를 찾았는가?

Answer
→ Faithfulness
→ 답변이 Context에 근거하는가?
```

> **RAG를 개선했다면, 같은 질문 세트로 개선 전후를 비교할 수 있어야 한다.**


# 1. Golden Test Set

RAG를 평가하려면 먼저 **정답 기준이 있는 질문 세트**가 필요하다.

이를 Golden Test Set이라고 한다.

이번 실습에서는 각 질문에 다음 두 정보만 둔다.

```text
question
→ 평가할 질문

relevant_page
→ 정답 근거가 있는 페이지
```

실제 프로젝트에서는 질문 수를 더 늘리면 된다.


In [1]:
# 작은 Golden Test Set
# 질문과 정답 근거가 있는 페이지를 함께 저장한다.
golden_set = [
    {
        "question": "기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?",
        "relevant_page": 17,
    },
    {
        "question": "2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?",
        "relevant_page": 18,
    },
    {
        "question": "2026년 AI 규제와 정책 환경은 어떤 방향으로 변화할 것으로 전망되나요?",
        "relevant_page": 19,
    },
]

for item in golden_set:
    print(item)


{'question': '기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?', 'relevant_page': 17}
{'question': '2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?', 'relevant_page': 18}
{'question': '2026년 AI 규제와 정책 환경은 어떤 방향으로 변화할 것으로 전망되나요?', 'relevant_page': 19}


# 2. 환경 설정

09번의 목적은 검색 코드를 다시 배우는 것이 아니다.

따라서 이 실습에서는 기존 ChromaDB를 이용해 간단히 검색한다.

실제 프로젝트에서는 아래 `retrieve()` 부분을
05~06번에서 만든 최종 Retrieval Pipeline으로 바꾸면 된다.


In [2]:
import sys
import re
from pathlib import Path

from langchain_chroma import Chroma

sys.path.append("..")
from common_config import embedding_model, llm_connect


# 02번에서 만든 ChromaDB 위치
DB_PATH = Path("chroma_db/ai_trend_report_docling")
COLLECTION_NAME = "ai_trend_report_2026_docling"

if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_ai_trend.ipynb를 먼저 실행하세요."
    )

# 임베딩 모델과 LLM 준비
embeddings = embedding_model()

llm = llm_connect(
    temperature=0,
    max_tokens=400,
)

# 기존 ChromaDB 연결
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)


def retrieve(question, k=3):
    # 평가할 검색 결과 Top-k를 가져온다.
    # 실제 프로젝트에서는 이 부분을 최종 Retrieval Pipeline으로 교체하면 된다.
    return vector_store.similarity_search(
        question,
        k=k,
    )


# 3. Retrieval 평가 — Hit@K

`Hit@K`는 가장 이해하기 쉬운 검색 평가 지표이다.

질문은 하나이다.

> **정답 근거가 Top-K 안에 들어왔는가?**

예:

```text
정답 페이지: 18

검색 결과
1위 → p.12
2위 → p.18
3위 → p.21

Hit@3 = 1
```

정답이 Top-3 안에 없으면 `0`, 있으면 `1`이다.


In [3]:
def get_pages(doc):
    # metadata의 pages 또는 page에서 숫자만 꺼낸다.
    value = doc.metadata.get(
        "pages",
        doc.metadata.get("page", "")
    )

    return {
        int(x)
        for x in re.findall(r"\d+", str(value))
    }


hits = []

for item in golden_set:
    # 질문마다 Top-3 문서를 검색한다.
    docs = retrieve(
        item["question"],
        k=3,
    )

    # Top-3 안에 정답 페이지가 있는지 확인한다.
    hit = 0

    for doc in docs:
        if item["relevant_page"] in get_pages(doc):
            hit = 1
            break

    hits.append(hit)

    print("=" * 80)
    print("질문:", item["question"])
    print("정답 페이지:", item["relevant_page"])
    print("Hit@3:", hit)

# 전체 질문 중 몇 %를 맞췄는지 계산
hit_at_3 = sum(hits) / len(hits)

print("\n전체 Hit@3:", round(hit_at_3, 3))


질문: 기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?
정답 페이지: 17
Hit@3: 1
질문: 2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
정답 페이지: 18
Hit@3: 0
질문: 2026년 AI 규제와 정책 환경은 어떤 방향으로 변화할 것으로 전망되나요?
정답 페이지: 19
Hit@3: 1

전체 Hit@3: 0.667


# 4. Answer 평가 — Faithfulness

검색이 잘되어도 최종 답변이 Context를 벗어나면 좋은 RAG라고 보기 어렵다.

그래서 답변에서는 **Faithfulness**를 확인한다.

질문은 하나이다.

> **답변 내용이 제공된 Context에서 뒷받침되는가?**

이번 실습에서는 LLM-as-a-Judge를 이용해 1~5점으로 평가한다.

```text
5점
→ 거의 모든 내용이 Context에 근거함

3점
→ 일부는 근거가 있지만 불명확한 내용이 있음

1점
→ Context와 거의 관계없이 답함
```


In [1]:
# 평가 예제로 Golden Set의 첫 번째 질문을 사용한다.
question = golden_set[0]["question"]

# 관련 문서 3개 검색
docs = retrieve(
    question,
    k=3,
)

# 검색된 문서를 Context로 합친다.
context = ""

for i, doc in enumerate(docs, start=1):
    context += f"""
[문서 {i}]
{doc.page_content}

"""


# 먼저 Grounded Answer를 생성한다.
answer_prompt = f"""
아래 Context만 사용해서 질문에 답하세요.

규칙:
- Context에 있는 내용만 사용하세요.
- Context에 없는 내용은 추측하지 마세요.
- 간결하게 답하세요.

질문:
{question}

Context:
{context}
"""

answer_response = llm.invoke(answer_prompt)
answer = answer_response.content

print("[질문]")
print(question)

print("\n[답변]")
print(answer)


# 같은 LLM을 Judge로 사용해 Faithfulness를 1~5점으로 평가한다.
judge_prompt = f"""
다음 답변이 Context에 얼마나 충실한지 1~5점으로 평가하세요.

5 = 답변의 주요 내용이 Context에서 직접 뒷받침된다.
3 = 일부 내용은 근거가 있지만 일부는 불명확하다.
1 = 대부분 Context에서 근거를 찾기 어렵다.

숫자 하나만 출력하세요.

Context:
{context}

Answer:
{answer}
"""

judge_response = llm.invoke(judge_prompt)

# 응답에서 1~5 숫자 하나를 찾는다.
match = re.search(r"[1-5]", judge_response.content)

faithfulness = (
    int(match.group())
    if match
    else None
)

print("\nFaithfulness:", faithfulness, "/ 5")


NameError: name 'golden_set' is not defined

### 문제 : 위에 Golden test set을 모두 평가하기

In [ ]:
def evaluate_faithfulness(question, k=3):
    # 1. 관련 문서 Top-k 검색
    docs = retrieve(
        question,
        k=k,
    )

    # 2. 검색된 문서를 Context로 합친다.
    context = ""

    for i, doc in enumerate(docs, start=1):
        context += f"""
[문서 {i}]
{doc.page_content}

"""

    # 3. Grounded Answer 생성
    answer_prompt = f"""
아래 Context만 사용해서 질문에 답하세요.

규칙:
- Context에 있는 내용만 사용하세요.
- Context에 없는 내용은 추측하지 마세요.
- 간결하게 답하세요.

질문:
{question}

Context:
{context}
"""

    answer = llm.invoke(answer_prompt).content

    # 4. 같은 LLM을 Judge로 사용해 Faithfulness를 1~5점으로 평가
    judge_prompt = f"""
다음 답변이 Context에 얼마나 충실한지 1~5점으로 평가하세요.

5 = 답변의 주요 내용이 Context에서 직접 뒷받침된다.
3 = 일부 내용은 근거가 있지만 일부는 불명확하다.
1 = 대부분 Context에서 근거를 찾기 어렵다.

숫자 하나만 출력하세요.

Context:
{context}

Answer:
{answer}
"""

    judge_response = llm.invoke(judge_prompt)

    # 응답에서 1~5 숫자 하나를 찾는다.
    match = re.search(r"[1-5]", judge_response.content)

    faithfulness = (
        int(match.group())
        if match
        else None
    )

    return answer, faithfulness


scores = []

for item in golden_set:
    # 질문마다 답변을 생성하고 Faithfulness를 평가한다.
    answer, faithfulness = evaluate_faithfulness(
        item["question"],
        k=3,
    )

    # Judge가 숫자를 주지 않은 경우는 평균 계산에서 제외한다.
    if faithfulness is not None:
        scores.append(faithfulness)

    print("=" * 80)
    print("question :", item["question"])
    print("answer :", answer)
    print(f"Faithfulness : {faithfulness}/5")

# 전체 질문의 평균 Faithfulness 계산
if scores:
    avg_faithfulness = sum(scores) / len(scores)
    print("\n평균 Faithfulness:", round(avg_faithfulness, 2), "/ 5")
else:
    print("\n평가된 Faithfulness 점수가 없습니다.")+

### 예상 결과

> ⚠️ API 토큰 한도로 위 셀을 실제 실행하지 못해, 보고서 원문(p.17~19)을 기준으로 작성한 **예상 출력**이다. 실제 실행 시 답변 문장과 점수는 달라질 수 있다.

```text
================================================================================
question : 기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?
answer : 기업 내부에서 AI 에이전트는 문서 처리, 고객 지원, 운영 자동화 등에 활용이 증가하며, 사람–에이전트–시스템이 혼합된 업무 구조가 일부 영역에서 확산될 가능성이 있습니다. 또한 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물 것으로 평가됩니다.
Faithfulness : 5/5
================================================================================
question : 2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
answer : 2026년 AI 기술은 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 전망입니다. 합성데이터·추론형 AI·멀티모달 기술이 주요 경쟁 축으로 자리 잡고, 학습 효율 향상·복합 정보 처리·설명 가능성 강화 등 모델 내부 구조의 질적 개선이 이어질 것으로 예상됩니다. 아울러 안전성·정합성 검증의 중요성이 커지면서 신뢰성 평가와 표준화가 기술 확산의 기반 인프라로 자리 잡을 것으로 전망됩니다.
Faithfulness : 5/5
================================================================================
question : 2026년 AI 규제와 정책 환경은 어떤 방향으로 변화할 것으로 전망되나요?
answer : 2026년 AI 정책 환경은 규제를 제약이 아닌 성장을 위한 안전장치로 재정의하고, 글로벌 규제체계와의 정합성을 높이는 방향으로 재편될 전망입니다. EU AI Act 등과의 정합성을 위해 국내 AI 기본법의 시행령·가이드라인이 구체화되고, 의료·채용 등 고위험 AI의 안전성 검증과 제3자 인증이 필수화되며, 워터마크·딥페이크 탐지 기술이 법제화될 것으로 예상됩니다.
Faithfulness : 5/5

평균 Faithfulness: 5.0 / 5
```

**해석**

- 세 질문 모두 "Context만 사용" 규칙으로 답변을 생성하므로, 답변이 검색된 문서 범위를 벗어나지 않는 한 Faithfulness는 높게 나온다.
- 단, 3번 셀에서 두 번째 질문은 `Hit@3 = 0`이었다. 즉 정답 페이지(p.18)가 Top-3에 없었으므로, 실제 실행에서는 답변이 위 예상보다 빈약하거나 다른 페이지 내용으로 채워질 수 있다.
- 이 경우에도 답변이 검색된 Context에만 근거하면 Faithfulness는 높게 나온다. **Faithfulness는 "근거에 충실한가"만 보고 "정답인가"는 보지 않기 때문에**, Hit@K와 함께 봐야 한다.

# 5. 이번 실습에서 꼭 기억할 것

RAG 평가는 크게 두 부분으로 나눌 수 있다.

| 평가 대상 | 핵심 질문 | 이번 실습 지표 |
|---|---|---|
| Retrieval | 관련 근거를 찾았는가? | Hit@K |
| Answer | 답변이 근거에 충실한가? | Faithfulness |

이 두 가지를 이해하면 기본적인 RAG 평가 흐름은 충분하다.

```text
Golden Test Set
      ↓
Retrieval
      ↓
Hit@K
      ↓
Answer 생성
      ↓
Faithfulness
```


# 6. 심화 지표는 나중에

실무에서는 다음 지표도 많이 사용한다.

```text
MRR
→ 정답 근거가 얼마나 높은 순위에 있는가?

Answer Relevance
→ 답변이 질문에 직접적으로 답하고 있는가?
```

하지만 이번 실습에서는 코드로 구현하지 않는다.

취업 준비 단계에서는 먼저 다음 두 개를 정확히 이해하는 것이 더 중요하다.

```text
Hit@K
Faithfulness
```


# 7. Advanced RAG 전체 흐름 정리

```text
01 Naive RAG
       ↓
02 Document Processing
       ↓
03 Query Rewriting
       ↓
04 Multi Query
       ↓
05 Dense + BM25 + RRF
       ↓
06 Reranking
       ↓
07 Context Engineering
       ↓
08 Grounded Answer
       ↓
09 Evaluation
```

## 09번에서 기억할 한 문장

> **RAG를 개선했다고 말하려면 같은 Golden Test Set으로 개선 전후를 비교해야 한다.**

프로젝트에서는 다음 정도의 결과표만 있어도 충분하다.

```text
                Naive     Advanced
Hit@3            0.xx        0.xx
Faithfulness     x/5         x/5
```

기능을 많이 넣는 것보다  
**개선 효과를 측정할 수 있는 구조**를 만드는 것이 더 중요하다.
